# 第 5 章 · 用 Python 写节点：发布、订阅、服务、参数

本章回答：一个 ROS 2 节点的 Python 代码长什么样？定时器、发布者、订阅者、服务、客户端、参数分别怎么写？为什么"在回调里同步调用服务"会卡死？

配套讲义：仓库里 `ros2/notes/lessons/05-python-nodes.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 rclpy。这里的 MiniNode / MiniExecutor 只模仿真 API 的形状（create_timer、create_publisher、create_subscription、create_service、call_async、declare_parameter），时间是假的
（按事件推进，不真的 sleep），所以 0.5 秒的定时器跑 2 秒只要一眨眼。真代码见讲义第 1–4 步。

In [ ]:
import heapq
import itertools
from typing import ClassVar

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 骨架 | 不 spin 回调不跑 | 建好定时器后看 `counter` | 0 条 |
| 第 2 步 定时器 + 发布订阅 | 每 0.5 秒发一条 | `ex.spin_for(2.0)` | 收到 `Hello World: 0`…`3` 共 4 条 |
| 第 3 步 future | call_async 立刻返回凭条 | `fut.done` | 先 False，spin 后 True，sum = 42 |
| 第 3 步 死锁 | 单线程回调里同步 call | `node.call(...)` 在定时器里 | `DeadlockError` |
| 第 4 步 参数 | 先声明；默认值定类型 | `set_parameter('my_parameter', 42)` | False（被拒绝） |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `DeadlockError`**：单线程 executor 里同步等回复：回调等回复，回复等回调结束。

In [ ]:
class DeadlockError(RuntimeError):
    """单线程 executor 里同步等回复：回调等回复，回复等回调结束。"""

**类 `Future`**：凭条：先拿到，结果之后才填进去；可以挂"填好以后做什么"的回调。

In [ ]:
class Future:
    """凭条：先拿到，结果之后才填进去；可以挂"填好以后做什么"的回调。"""

    def __init__(self):
        self.done = False
        self._result = None
        self._callbacks = []

    def set_result(self, value):
        self.done, self._result = True, value
        for cb in self._callbacks:
            cb(self)

    def result(self):
        return self._result

    def add_done_callback(self, cb):
        self._callbacks.append(cb)

**类 `MiniExecutor`**：单线程 executor：手里一张按时间排的待办表，spin 时一件一件取出来调回调。

In [ ]:
class MiniExecutor:
    """单线程 executor：手里一张按时间排的待办表，spin 时一件一件取出来调回调。"""

    def __init__(self):
        self.now = 0.0
        self._todo = []
        self._seq = itertools.count()
        self.in_callback = False

    def schedule(self, t: float, fn):
        heapq.heappush(self._todo, (t, next(self._seq), fn))

    def spin_for(self, seconds: float):
        end = self.now + seconds
        while self._todo and self._todo[0][0] <= end + 1e-9:
            t, _, fn = heapq.heappop(self._todo)
            self.now = t
            self.in_callback = True
            try:
                fn()
            finally:
                self.in_callback = False
        self.now = end

    def spin_until_future_complete(self, future: Future, timeout: float = 10.0):
        end = self.now + timeout
        while not future.done and self._todo and self._todo[0][0] <= end:
            self.spin_for(self._todo[0][0] - self.now)

**类 `MiniNode`**：模仿 rclpy.node.Node 的几个方法。所有节点共用一个 executor 和一张"话题 / 服务"表。

In [ ]:
class MiniNode:
    """模仿 rclpy.node.Node 的几个方法。所有节点共用一个 executor 和一张"话题 / 服务"表。"""

    topics: ClassVar[dict[str, list]] = {}
    services: ClassVar[dict[str, object]] = {}

    def __init__(self, name: str, executor: MiniExecutor):
        self.name, self.ex = name, executor
        self._params: dict[str, object] = {}
        self._post_set = []
        self.log: list[str] = []

    # —— 定时器、发布、订阅 ——
    def create_timer(self, period: float, callback):
        def fire():
            callback()
            self.ex.schedule(self.ex.now + period, fire)

        self.ex.schedule(self.ex.now + period, fire)

    def create_publisher(self, topic: str, depth: int = 10):
        node = self

        class Pub:
            def publish(self, msg):
                for cb in node.topics.get(topic, []):
                    node.ex.schedule(node.ex.now, lambda cb=cb: cb(msg))  # 不立刻调：等 spin 取出来

        return Pub()

    def create_subscription(self, topic: str, callback, depth: int = 10):
        self.topics.setdefault(topic, []).append(callback)

    # —— 服务 ——
    def create_service(self, name: str, callback):
        self.services[name] = callback

    def call_async(self, name: str, request: dict) -> Future:
        """立刻返回凭条；回复要等 executor 在 spin 里处理服务端回调后才填上。"""
        fut = Future()
        handler = self.services[name]
        self.ex.schedule(self.ex.now + 0.001, lambda: fut.set_result(handler(request, {})))
        return fut

    def call(self, name: str, request: dict):
        """同步调用：在单线程 executor 的回调里用 → 没人去处理回复，卡死。"""
        fut = self.call_async(name, request)
        if self.ex.in_callback:
            raise DeadlockError("唯一的线程正卡在这个回调里等回复，可回复要这个线程空下来才能处理")
        self.ex.spin_until_future_complete(fut)
        return fut.result()

    # —— 参数 ——
    def declare_parameter(self, name: str, default):
        self._params[name] = default

    def get_parameter(self, name: str):
        if name not in self._params:
            raise KeyError(f"参数 {name} 没声明")
        return self._params[name]

    def set_parameter(self, name: str, value) -> bool:
        """类型必须和默认值一致；成功后调 post-set 回调。返回是否成功。"""
        if name not in self._params:
            raise KeyError(f"参数 {name} 没声明")
        if type(value) is not type(self._params[name]):
            return False
        self._params[name] = value
        for cb in self._post_set:
            cb(name, value)
        return True

    def add_post_set_parameters_callback(self, cb):
        self._post_set.append(cb)

**函数 `add_two_ints`**：服务端回调：收到请求、填好回复、返回。

In [ ]:
def add_two_ints(request: dict, response: dict) -> dict:
    """服务端回调：收到请求、填好回复、返回。"""
    response["sum"] = request["a"] + request["b"]
    return response

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
MiniNode.topics.clear()
MiniNode.services.clear()

In [ ]:
print("\n第一步 · 节点骨架：不 spin，回调一次都不跑")
ex = MiniExecutor()
talker = MiniNode("minimal_publisher", ex)
pub = talker.create_publisher("topic", 10)
counter = {"i": 0}
def timer_callback():
    pub.publish({"data": f"Hello World: {counter['i']}"})
    counter["i"] += 1
talker.create_timer(0.5, timer_callback)
print("  建好定时器，还没 spin：发了", counter["i"], "条")

In [ ]:
print("\n第二步 · 定时器每 0.5 秒发一条，订阅者收到就记下来")
listener = MiniNode("minimal_subscriber", ex)
listener.create_subscription("topic", lambda msg: listener.log.append(msg["data"]), 10)
ex.spin_for(2.0)
print("  spin 2 秒（假时间）后，订阅者收到:", listener.log)

In [ ]:
print("\n第三步 · 服务：call_async 先拿凭条，spin_until_future_complete 等它填上")
server = MiniNode("minimal_service", ex)
server.create_service("add_two_ints", add_two_ints)
client = MiniNode("minimal_client_async", ex)
fut = client.call_async("add_two_ints", {"a": 41, "b": 1})
print("  刚拿到凭条: done =", fut.done)
ex.spin_until_future_complete(fut)
print("  spin 之后: done =", fut.done, " sum =", fut.result()["sum"])

In [ ]:
print("\n第三步（续）· 在定时器回调里同步 call()：单线程 executor 卡死")
ex2 = MiniExecutor()
node2 = MiniNode("deadlock_demo", ex2)
errors = []
def bad_timer():
    try:
        node2.call("add_two_ints", {"a": 1, "b": 2})
    except DeadlockError as e:
        errors.append(str(e))
node2.create_timer(1.0, bad_timer)
ex2.spin_for(1.0)
print("  卡死原因:", errors[0])

In [ ]:
print("\n第四步 · 参数：先声明；默认值决定类型；改成功后 post-set 回调更新成员变量")
p = MiniNode("minimal_param_node", ex)
p.declare_parameter("my_parameter", "world")
p.add_post_set_parameters_callback(lambda name, value: p.log.append(f"{name} 改成了 {value}"))
print("  get →", p.get_parameter("my_parameter"))
print("  set 'earth' →", p.set_parameter("my_parameter", "earth"), "；post-set 回调记下:", p.log)
print("  set 42（整数）→", p.set_parameter("my_parameter", 42), "（类型不对，被拒绝）")
try:
    p.get_parameter("my_paramter")
except KeyError as e:
    print("  拼错名字 →", e)

## 3. 代码里才看得到的

1. **这是个迷你 rclpy**：方法名和真 API 一样，但时间是假的（`MiniExecutor` 按待办表推进，不 sleep），所以 2 秒瞬间跑完。
2. **发布不等于回调**：`publish` 只是往 executor 的待办表里加一件事，真正调订阅回调要等 spin，和第 1 章"spin 才叫醒回调"一致。
3. **死锁在真 rclpy 里不会报错，而是一直卡住**。模拟里为了能演示，检测到"在回调里同步等"就抛异常。
4. **参数校验回调（on_set）这里没写**，留给练习 5-4。

## 4. 自己改一改

- 把定时器周期改成 0.2 秒，`spin_for(1.0)`，数一数收到几条；
- 把 `call_async` 的请求改成别的两个数，看 `sum`。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **骨架**：继承 Node 起个名，init、spin 加 Ctrl+C。
- **四件套**：`create_publisher / create_subscription / create_timer / create_service`，对象都挂在 `self` 上。
- **服务**：服务端填 response 返回；客户端 `call_async` 拿凭条，回调里别用同步 `call()`。
- **参数**：先声明再用；校验在 on-set（无副作用），收尾在 post-set。